## Import Data

In [1]:
import kagglehub
import shutil
import os

# Download to kaggle cache
cache_path = kagglehub.dataset_download("jyronw/us-state-of-the-union-addresses-1790-2019")

# Copy files to local data/ directory
data_dir = os.path.join(os.path.dirname(os.path.abspath("Visualization.ipynb")))
os.makedirs(data_dir, exist_ok=True)
for file in os.listdir(cache_path):
    shutil.copy2(os.path.join(cache_path, file), data_dir)

print("Data saved to:", data_dir)

Data saved to: /Users/dkhun/Documents/Code/CS4990-Projects/Project2


## Interactive Word Cloud — First 5 Presidents

In [2]:
import pandas as pd
import ast
import re
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from wordcloud import WordCloud, STOPWORDS

# Load data
df = pd.read_csv("data/state_ofthe_union_texts.csv")

# Clean president names (some entries have "by " prefix)
df["President"] = df["President"].str.replace(r"^by\s+", "", regex=True).str.strip()

# Get first 5 unique presidents in chronological order
first_5 = df["President"].unique()[:5].tolist()

def get_text_for_president(name):
    rows = df[df["President"] == name]
    combined = []
    for raw in rows["Text"]:
        try:
            paragraphs = ast.literal_eval(raw)
            combined.extend(paragraphs)
        except Exception:
            combined.append(str(raw))
    return " ".join(combined)

def clean_text(text):
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return text.lower()

stopwords = set(STOPWORDS) | {
    "will", "shall", "may", "must", "can", "upon", "would", "also",
    "one", "two", "three", "us", "great", "every", "made", "said",
    "make", "much", "now", "yet", "well", "mr", "th", "ye", "hath",
}

dropdown = widgets.Dropdown(
    options=first_5,
    value=first_5[0],
    description="President:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

output = widgets.Output()

def update_wordcloud(change):
    with output:
        clear_output(wait=True)
        president = change["new"]
        text = clean_text(get_text_for_president(president))
        wc = WordCloud(
            width=900,
            height=500,
            background_color="white",
            stopwords=stopwords,
            max_words=150,
            colormap="viridis",
            collocations=False,
        ).generate(text)
        fig, ax = plt.subplots(figsize=(13, 7))
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Word Cloud — {president}'s State of the Union Addresses", fontsize=16, pad=14)
        plt.tight_layout()
        plt.show()

dropdown.observe(update_wordcloud, names="value")

display(widgets.VBox([dropdown, output]))
update_wordcloud({"new": first_5[0]})

FileNotFoundError: [Errno 2] No such file or directory: 'data/state_ofthe_union_texts.csv'